# Clase 4 — GraphRAG

## RAG con grafos y knowledge graphs

**Objetivos de la clase:**
- Entender por qué el RAG vectorial falla en preguntas que requieren razonar sobre **relaciones entre entidades** (multi-hop).
- Construir un knowledge graph en **Neo4j** a partir de datos estructurados.
- Comparar recuperación **vectorial** vs recuperación por **grafo** (Cypher) sobre las mismas preguntas.
- Combinar ambas en una sola consulta con `retrieval_query`.

📎 Material de referencia: `Clase_GraphRag.pdf` (completo). La detección de comunidades y la recuperación local/global (slide "Recuperación local vs global vs híbrida") quedan como enfoque **opcional** — no se implementan en este notebook.

## 1. El límite del RAG vectorial: preguntas multi-hop

Considera esta pregunta (slide "RAG clásico: flujo y límites"):

> *"¿Qué nombre tiene el hijo del inventor de la teoría de la relatividad?"*

Para responderla, un sistema necesita **dos saltos de razonamiento**: (1) identificar que Einstein inventó la teoría de la relatividad, y (2) buscar quién es el hijo de Einstein. Un retriever vectorial busca *un* chunk parecido a la pregunta completa — pero es poco probable que exista un chunk que mencione ambos hechos juntos. El resultado: respuestas incompletas o alucinadas.

Más adelante en este notebook vamos a toparnos con el mismo problema, pero con un ejemplo muy concreto: preguntas de **conteo y agregación** ("¿cuántos artículos publicó X?"), que la similitud semántica tampoco puede resolver.

## 2. Knowledge Graphs como memoria flexible

Un **knowledge graph (KG)** representa datos como elementos conectados, sin esquema rígido (slide "Knowledge Graphs como memoria flexible en RAG"):

- **Nodos**: entidades (personas, organizaciones, conceptos...) o documentos/chunks.
- **Aristas**: relaciones tipadas y con peso (co-mención, relación extraída, similitud, citación).
- **Atributos**: fuente, spans, confianza, embeddings.
- **Comunidades**: agrupaciones multi-nivel que capturan temas o cohesión.

**GraphRAG** (slide "¿Qué es Graph RAG?") extrae este grafo desde el corpus y lo usa para recuperar evidencia y razonar sobre conexiones. La arquitectura tipo Microsoft GraphRAG indexa en 4 pasos: *segmentación → extracción de entidades/relaciones → agrupación jerárquica (comunidades) → resúmenes por comunidad*. Nosotros vamos a trabajar con una versión más simple: datos ya estructurados (investigadores, artículos, temas) cargados directamente como grafo en Neo4j.

In [1]:
# Instalar los paquetes necesarios
!pip install -qU neo4j langchain langchain_openai langchain-community langchain_classic langchain-huggingface langchain-neo4j \
    python-dotenv==1.1.0 "networkx>=3.0" ipywidgets pandas matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

### Configurar Neo4j AuraDB (gratis)

1. Crea una cuenta gratis en [neo4j.com/cloud/aura-free](https://neo4j.com/cloud/aura-free/).
2. Crea una nueva instancia **AuraDB Free** (alcanza hasta 200k nodos / 400k relaciones, slide "Implementación con Neo4j").
3. Guarda las credenciales que te entrega al crearla: **URI**, **usuario** y **password**.
4. En Colab, agrégalas como *Secrets* (ícono 🔑 en el panel izquierdo): `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, tu `OPENROUTER_API_KEY` (para el LLM) y tu `OPENAI_API_KEY_DIPLOMADO` habitual (solo para los embeddings — OpenRouter no expone ese endpoint).

In [1]:
import os
from google.colab import userdata

def obtener_secret(nombre):
    try:
        valor = userdata.get(nombre)
    except userdata.SecretNotFoundError:
        valor = None
    return valor if valor else input(f"Ingresa {nombre}: ")

# OpenRouter para el LLM
os.environ["OPENROUTER_API_KEY"] = obtener_secret("OPENROUTER_API_KEY")
# La API key de OpenAI ya no hace falta: los embeddings de esta clase pasaron a HuggingFace
# os.environ["OPENAI_API_KEY"] = obtener_secret("OPENAI_API_KEY_DIPLOMADO")
os.environ["NEO4J_URI"] = obtener_secret("NEO4J_URI")
os.environ["NEO4J_USERNAME"] = obtener_secret("NEO4J_USERNAME")
os.environ["NEO4J_PASSWORD"] = obtener_secret("NEO4J_PASSWORD")

# HF_TOKEN es opcional: solo sube el rate limit de descarga de modelos, no es indispensable
hf_token = obtener_secret("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN cargado desde secrets.")
else:
    print("HF_TOKEN no encontrado — los embeddings de HuggingFace van a correr sin autenticación.")

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "deepseek/deepseek-v4-flash"  # cámbialo por cualquier modelo de openrouter.ai/models

def crear_llm(model=None, **kwargs):
    """ChatOpenAI apuntando a OpenRouter en vez de a la API de OpenAI directamente."""
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(
        model=model or OPENROUTER_MODEL,
        base_url=OPENROUTER_BASE_URL,
        api_key=os.environ["OPENROUTER_API_KEY"],
        **kwargs,
    )

ModuleNotFoundError: No module named 'google.colab'

### Cargar el dataset en el grafo

Usamos un dataset sintético de investigadores, artículos y temas (del cookbook https://huggingface.co/learn/cookbook/en/rag_with_knowledge_graphs_neo4j), pensado justamente para tener relaciones claras de coautoría y colaboración — ideal para ilustrar GraphRAG.

In [3]:
from langchain_community.graphs import Neo4jGraph
from neo4j import GraphDatabase

def detectar_database_neo4j():
    """Neo4jGraph() asume que la base de datos se llama 'neo4j', pero no siempre es así
    (algunas instancias Aura usan otro nombre). En vez de adivinar, le preguntamos al
    propio servidor: la base especial 'system' siempre existe y sabe qué bases hay."""
    driver = GraphDatabase.driver(
        os.environ["NEO4J_URI"],
        auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
    )
    try:
        with driver.session(database="system") as session:
            bases = [dict(r) for r in session.run("SHOW DATABASES")]
    finally:
        driver.close()
    return bases

bases = detectar_database_neo4j()
print("Bases de datos que ve tu instancia Neo4j:")
for b in bases:
    print(f"  - {b['name']!r} (estado: {b.get('currentStatus')})")

candidatas = [b["name"] for b in bases if b.get("currentStatus") == "online" and b["name"] != "system"]
NEO4J_DATABASE = "neo4j" if "neo4j" in candidatas else (candidatas[0] if candidatas else "neo4j")
print(f"\nUsando base de datos: {NEO4J_DATABASE!r}")

graph = Neo4jGraph(database=NEO4J_DATABASE)

q_cargar_articulos = """
LOAD CSV WITH HEADERS
FROM 'https://raw.githubusercontent.com/dcarpintero/generative-ai-101/main/dataset/synthetic_articles.csv'
AS row
FIELDTERMINATOR ';'
MERGE (a:Article {title:row.Title})
SET a.abstract = row.Abstract,
    a.publication_date = date(row.Publication_Date)
FOREACH (researcher in split(row.Authors, ',') |
    MERGE (p:Researcher {name:trim(researcher)})
    MERGE (p)-[:PUBLISHED]->(a))
FOREACH (topic in [row.Topic] |
    MERGE (t:Topic {name:trim(topic)})
    MERGE (a)-[:IN_TOPIC]->(t))
"""

graph.query(q_cargar_articulos)
print("Datos cargados en Neo4j.")

/tmp/ipykernel_616/1479296000.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.graphs import Neo4jGraph


Bases de datos que ve tu instancia Neo4j:
  - 'c4d06201' (estado: online)
  - 'system' (estado: online)
  - 'system' (estado: online)
  - 'system' (estado: online)

Usando base de datos: 'c4d06201'


/tmp/ipykernel_616/1479296000.py:28: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(database=NEO4J_DATABASE)


Datos cargados en Neo4j.


In [4]:
graph.refresh_schema()
print(graph.get_schema)

Node properties:
Article {title: STRING, abstract: STRING, publication_date: DATE, embedding: LIST}
Researcher {name: STRING}
Topic {name: STRING}
Community {id: INTEGER, summary: STRING}
Relationship properties:

The relationships:
(:Article)-[:IN_TOPIC]->(:Topic)
(:Researcher)-[:PUBLISHED]->(:Article)
(:Researcher)-[:BELONGS_TO]->(:Community)


## 3. Recuperación por similitud (vector index en Neo4j)

Igual que con FAISS en las clases anteriores, podemos construir un **índice vectorial** dentro de Neo4j: cada `Article` se embebe (topic + title + abstract) y se puede buscar por similitud. La diferencia es que este índice vive en la misma base de datos que el grafo, por lo que en principio se puede combinar con travesías de grafo (slide "Implementación con Neo4j").

In [5]:
from langchain_neo4j import Neo4jVector # Updated import path for Neo4jVector
# from langchain_neo4j.vectorstores import Neo4jVector # Deprecated path
# from langchain_community.vectorstores import Neo4jVector # Deprecated
# from langchain_community.embeddings import HuggingFaceEmbeddings # Deprecated
from langchain_huggingface import HuggingFaceEmbeddings # Updated import

# Initialize HuggingFaceEmbeddings
# You can choose a different model if needed, e.g., 'sentence-transformers/all-MiniLM-L6-v2'
hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_index = Neo4jVector.from_existing_graph(
    embedding=hf_embeddings, # Using HuggingFaceEmbeddings
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    index_name="articles",
    node_label="Article",
    text_node_properties=["topic", "title", "abstract"],
    embedding_node_property="embedding",
    database=NEO4J_DATABASE # Added to specify the correct database
)
print("Índice vectorial creado.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Índice vectorial creado.


### Armando la cadena RAG con las piezas "oficiales" de LangChain

En las clases 1-3 construimos el pipeline RAG a mano: `rag_prompt | llm | StrOutputParser()`, juntando los chunks recuperados con `"\n\n".join(...)` antes de pasarlos al prompt. Acá usamos las piezas que LangChain ofrece para ese mismo patrón, ya armadas:

- **`create_stuff_documents_chain(llm, prompt)`**: crea una mini-cadena que toma una lista de documentos y los "amontona" (*stuff* = rellenar) dentro del `{context}` del prompt, uno tras otro, antes de mandarlo al LLM. Es la estrategia de combinación más simple que existe — para corpus muy grandes que no caben en una sola llamada se usan otras, como `map_reduce` (resumir por partes y combinar).
- **`create_retrieval_chain(retriever, document_chain)`**: pega el retriever (que busca los documentos) con la mini-cadena anterior, armando el pipeline completo: **pregunta → retrieval → prompt con contexto → LLM → respuesta**. Es el reemplazo moderno de `RetrievalQA.from_chain_type(...)`, que LangChain está retirando.

Dos detalles de nomenclatura que vale la pena marcar, porque son distintos a lo que usamos en clases anteriores y ya nos generaron un par de bugs:

1. El prompt usa `{input}` (la pregunta) y `{context}` (los documentos recuperados) — son los nombres que estas funciones esperan por convención, no `{pregunta}`/`{contexto}` como en nuestras cadenas manuales.
2. El resultado de `vector_qa.invoke(...)` es un diccionario con `answer` (la respuesta final), `context` (la lista de documentos que se usaron) e `input` (la pregunta original) — **no** `result`, que era la clave de la `RetrievalQA` vieja. Por eso los widgets de abajo leen `r["answer"]` y `r["context"]`, no `r["result"]`.

In [6]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

llm = crear_llm(temperature=0);

# Define the prompt for the RAG chain
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the user's questions based on the below context:\n\n{context}"),
    ("user", "{input}"),
])

document_chain = create_stuff_documents_chain(llm, rag_prompt)

vector_qa = create_retrieval_chain(vector_index.as_retriever(), document_chain)


In [7]:
import ipywidgets as widgets

@widgets.interact_manual(
    pregunta=widgets.Textarea(
        value="Which articles discuss how AI might affect our daily life? Include titles and abstracts.",
        description="Pregunta",
        layout=widgets.Layout(width="100%", height="60px"),
    )
)
def preguntar_vector(pregunta):
    try:
        vector_qa
    except NameError:
        print("⚠️ Falta correr la celda de arriba que define `vector_qa` (sección 3). Ejecútala primero.")
        return
    r = vector_qa.invoke({"input": pregunta})
    print(f"--- {len(r['context'])} documentos recuperados por similitud ---")
    for i, doc in enumerate(r["context"], start=1):
        print(f"[{i}] {doc.page_content.strip()}\n")
    print("--- Respuesta del LLM ---")
    print(r["answer"])

interactive(children=(Textarea(value='Which articles discuss how AI might affect our daily life? Include title…

### El límite: preguntas de conteo y agregación

RAG vectorial es muy bueno para encontrar artículos **temáticamente parecidos** a la pregunta. Pero, ¿qué pasa si le preguntamos algo que requiere **contar** o **agregar** sobre el grafo completo?

In [8]:
@widgets.interact_manual(
    pregunta=widgets.Textarea(
        value="How many articles has Emily Chen published?",
        description="Pregunta",
        layout=widgets.Layout(width="100%", height="60px"),
    )
)
def preguntar_vector_conteo(pregunta):
    try:
        vector_qa
    except NameError:
        print("⚠️ Falta correr la celda de arriba que define `vector_qa` (sección 3). Ejecútala primero.")
        return
    r = vector_qa.invoke({"input": pregunta})
    print(f"--- {len(r['context'])} documentos recuperados por similitud (ninguno tiene por qué ser de Emily Chen) ---")
    for i, doc in enumerate(r["context"], start=1):
        print(f"[{i}] {doc.page_content.strip()}\n")
    print("--- Respuesta del LLM ---")
    print(r["answer"])
    print("\n(La respuesta correcta es 7 — compara con la sección 4 más abajo.)")

interactive(children=(Textarea(value='How many articles has Emily Chen published?', description='Pregunta', la…

El retriever vectorial solo trae los `k` artículos más *parecidos semánticamente* a la pregunta — no tiene forma de "contar todos los artículos de Emily Chen", porque eso requiere recorrer **todo** el grafo, no solo los vecinos más cercanos en el espacio de embeddings.

## 4. Recuperación por grafo (Cypher)

`GraphCypherQAChain` traduce la pregunta en lenguaje natural a una consulta **Cypher** (el lenguaje de consultas de Neo4j, inspirado en SQL), la ejecuta contra el grafo completo, y usa el resultado como contexto para generar la respuesta final (slide "Implementación con Neo4j").

In [9]:
from langchain_neo4j import GraphCypherQAChain

graph.refresh_schema()

cypher_chain = GraphCypherQAChain.from_llm(
    cypher_llm=crear_llm(model="openai/gpt-4o", temperature=0),
    qa_llm=crear_llm(model="openai/gpt-4o", temperature=0),
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True,  # Cypher puede modificar datos: lo habilitamos explícitamente porque confiamos en este grafo de prueba
)


In [10]:
@widgets.interact_manual(
    pregunta=widgets.Textarea(
        value="How many articles has Emily Chen published?",
        description="Pregunta",
        layout=widgets.Layout(width="100%", height="60px"),
    )
)
def preguntar_grafo(pregunta):
    try:
        cypher_chain
    except NameError:
        print("⚠️ Falta correr la celda de arriba que define `cypher_chain` "
              "(la de `GraphCypherQAChain.from_llm(...)`, bajo el título '## 4. Recuperación por grafo'). "
              "Ejecútala primero y vuelve a intentar este widget.")
        return
    r = cypher_chain.invoke({"query": pregunta})
    print(r["result"])

interactive(children=(Textarea(value='How many articles has Emily Chen published?', description='Pregunta', la…

Con `verbose=True` puedes ver, en la salida de la celda anterior, la consulta Cypher generada — por ejemplo, para "¿Cuántos artículos publicó Emily Chen?" debería generar algo como:

```cypher
MATCH (r:Researcher {name: "Emily Chen"})-[:PUBLISHED]->(a:Article)
RETURN COUNT(a)
```

Prueba también estas preguntas (de la slide "Query Samples using Natural Language"):

In [11]:
for pregunta in [
    "Are there any pair of researchers who have published more than three articles together?",
    "Which researcher has collaborated with the most peers?",
]:
    print("Pregunta:", pregunta)
    print(cypher_chain.invoke({"query": pregunta})["result"])
    print()

Pregunta: Are there any pair of researchers who have published more than three articles together?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (r1:Researcher)-[:PUBLISHED]->(a:Article)<-[:PUBLISHED]-(r2:Researcher)
WHERE r1 <> r2
WITH r1, r2, COUNT(a) AS shared_articles
WHERE shared_articles > 3
RETURN r1.name, r2.name, shared_articles

Full Context:
[{'r1.name': 'David Johnson', 'r2.name': 'Emily Chen', 'shared_articles': 4}, {'r1.name': 'Robert Taylor', 'r2.name': 'Emily Chen', 'shared_articles': 4}, {'r1.name': 'Emily Chen', 'r2.name': 'David Johnson', 'shared_articles': 4}, {'r1.name': 'Emily Chen', 'r2.name': 'Robert Taylor', 'shared_articles': 4}]

> Finished chain.
Yes, David Johnson and Emily Chen, as well as Robert Taylor and Emily Chen, have published more than three articles together.

Pregunta: Which researcher has collaborated with the most peers?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (r:Researcher)-[:PUBLISHED]->(:Art

## 5. Comparación final: vectorial vs grafo en preguntas multi-hop

In [12]:
preguntas_multihop = [
    "How many articles has Emily Chen published?",
    "Are there any pair of researchers who have published more than three articles together?",
    "Which researcher has collaborated with the most peers?",
]

filas = []
for p in preguntas_multihop:
    resp_vector = vector_qa.invoke({"input": p})["answer"]
    resp_grafo = cypher_chain.invoke({"query": p})["result"]
    filas.append({"pregunta": p, "RAG vectorial": resp_vector, "GraphRAG (Cypher)": resp_grafo})

import pandas as pd
pd.set_option("display.max_colwidth", None)
pd.DataFrame(filas)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (r:Researcher {name: "Emily Chen"})-[:PUBLISHED]->(a:Article)
RETURN COUNT(a) AS numberOfArticles

Full Context:
[{'numberOfArticles': 7}]



> Finished chain.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (r1:Researcher)-[:PUBLISHED]->(a:Article)<-[:PUBLISHED]-(r2:Researcher)
WHERE r1 <> r2
WITH r1, r2, COUNT(a) AS sharedArticles
WHERE sharedArticles > 3
RETURN r1.name, r2.name, sharedArticles

Full Context:
[{'r1.name': 'David Johnson', 'r2.name': 'Emily Chen', 'sharedArticles': 4}, {'r1.name': 'Robert Taylor', 'r2.name': 'Emily Chen', 'sharedArticles': 4}, {'r1.name': 'Emily Chen', 'r2.name': 'David Johnson', 'sharedArticles': 4}, {'r1.name': 'Emily Chen', 'r2.name': 'Robert Taylor', 'sharedArticles': 4}]

> Finished chain.




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (r:Researcher)-[:PUBLISHED]->(:Article)<-[:PUBLISHED]-(peer:Researcher)
WITH r, COUNT(DISTINCT peer) AS peerCount
RETURN r.name AS researcher, peerCount
ORDER BY peerCount DESC
LIMIT 1

Full Context:
[{'researcher': 'David Johnson', 'peerCount': 6}]

> Finished chain.


,pregunta,RAG vectorial,GraphRAG (Cypher)
0,How many articles has Emily Chen published?,"The provided context does not mention any articles authored by Emily Chen, so it is not possible to determine how many articles she has published based solely on this information.",Emily Chen has published 7 articles.
1,Are there any pair of researchers who have published more than three articles together?,"Based on the provided context, there is no information about individual researchers or their collaborative publication history. The context only includes titles and abstracts of research topics, not author names or co-authorship data. Therefore, I cannot determine if any pair of researchers have published more than three articles together.","Yes, David Johnson and Emily Chen, as well as Robert Taylor and Emily Chen, have published more than three articles together."
2,Which researcher has collaborated with the most peers?,"Based on the provided context, there is no information about researchers or their collaborations. The context only includes topic titles and abstracts, with no mention of specific researchers or peer collaboration counts. Therefore, I cannot determine which researcher has collaborated with the most peers.","David Johnson has collaborated with the most peers, with a peer count of 6."


## 6. Combinando vector + grafo en una sola consulta (`retrieval_query`)

En la comparación anterior tratamos la recuperación vectorial y la recuperación por grafo como dos caminos separados, invocados por separado. Pero `Neo4jVector` acepta un parámetro, `retrieval_query`, que permite combinar ambas en **una sola consulta**: el índice vectorial encuentra los nodos más parecidos a la pregunta (igual que en la sección 3), y luego, para cada uno de esos nodos, se ejecuta una travesía Cypher que agrega contexto de grafo — sin necesitar un agente que elija entre dos herramientas separadas.

Es el mismo patrón que usan sistemas GraphRAG reales para no perder ni la búsqueda semántica ni las relaciones estructuradas: la similitud encuentra el punto de entrada (qué artículo es relevante para la pregunta), y el grafo expande el contexto alrededor de ese punto de entrada (quién lo escribió, con quién más colabora esa gente).

In [13]:
retrieval_query_hibrida = """
RETURN node.title + "\\n" + node.abstract AS text,
       score,
       {
           autores: [(node)<-[:PUBLISHED]-(r:Researcher) | r.name],
           tema: [(node)-[:IN_TOPIC]->(t:Topic) | t.name],
           colaboradores: [(node)<-[:PUBLISHED]-(:Researcher)-[:PUBLISHED]->(otro:Article)<-[:PUBLISHED]-(r2:Researcher) WHERE otro <> node | r2.name]
       } AS metadata
"""

vector_index_hibrido = Neo4jVector.from_existing_index(
    embedding=hf_embeddings,
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    index_name="articles",
    database=NEO4J_DATABASE,
    retrieval_query=retrieval_query_hibrida,
)
print("Retriever híbrido creado (mismo índice 'articles', con travesía de grafo agregada por nodo).")

Retriever híbrido creado (mismo índice 'articles', con travesía de grafo agregada por nodo).


In [14]:
@widgets.interact_manual(
    pregunta=widgets.Textarea(
        value="Which articles discuss how AI might affect our daily life, and who else might I want to talk to about that topic?",
        description="Pregunta",
        layout=widgets.Layout(width="100%", height="60px"),
    ),
    k=widgets.IntSlider(value=3, min=1, max=6, description="k"),
)
def preguntar_hibrido(pregunta, k):
    try:
        vector_index_hibrido
    except NameError:
        print("⚠️ Falta correr la celda de arriba que define `vector_index_hibrido`. Ejecútala primero.")
        return

    docs = vector_index_hibrido.similarity_search(pregunta, k=k)

    bloques = []
    for doc in docs:
        meta = doc.metadata
        bloques.append(
            f"{doc.page_content}\n"
            f"Autores: {', '.join(meta['autores']) or 'desconocido'}\n"
            f"Tema: {', '.join(meta['tema']) or 'desconocido'}\n"
            f"Colaboradores de los autores (en otros artículos): {', '.join(meta['colaboradores']) or 'ninguno'}"
        )
    contexto = "\n\n---\n\n".join(bloques)

    print(f"--- Contexto combinado ({len(docs)} artículos por similitud, cada uno enriquecido con travesía de grafo) ---")
    print(contexto)

    respuesta = crear_llm(temperature=0).invoke(
        f"Responde la pregunta usando solo este contexto:\n\n{contexto}\n\nPregunta: {pregunta}"
    ).content
    print("\n--- Respuesta del LLM ---")
    print(respuesta)

interactive(children=(Textarea(value='Which articles discuss how AI might affect our daily life, and who else …

## Ejercicios propuestos

1. **Nuevas preguntas en lenguaje natural**
   - Escribe 2-3 preguntas nuevas para `cypher_chain` (por ejemplo, sobre un tema específico o un rango de fechas) y revisa el Cypher generado con `verbose=True`.


2. **Extender el `retrieval_query` híbrido**
   - Modifica `retrieval_query_hibrida` (sección 6) para agregar otro dato del grafo al contexto — por ejemplo, la fecha de publicación (`node.publication_date`) o cuántos artículos ha publicado cada autor. ¿Cambia la respuesta del LLM al agregar más contexto de grafo?

## Cierre del curso

En estas 4 clases construimos, evaluamos y extendimos un sistema RAG completo:

1. **RAG Simple** — chunking, embeddings, retrieval, generación.
2. **RAG Avanzado** — hybrid search, reranking, filtrado de relevancia.
3. **Evaluación de RAG** — métricas de retrieval, generación y RAG-específicas.
4. **GraphRAG** — knowledge graphs, recuperación vectorial y por grafo (Cypher), combinación de ambas en una sola consulta, razonamiento multi-hop.

La elección entre estas técnicas (y entre RAG y fine-tuning, slide "Cuándo usar Fine-tuning y cuándo RAG") depende siempre del problema: qué tan dinámicos son los datos, qué tipo de preguntas se esperan, y qué costo de latencia/infraestructura es aceptable.